## Notebook 3: Visually Inspecting The Data

(Gill & Jer Clean/Re-upload Accepted Data)

In [1]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np

In [2]:
# Clone our BMEN 600 group's GitHub repository into Colab.
!git clone https://github.com/chelseataylor1/BMEN-600-Group-19.git

Cloning into 'BMEN-600-Group-19'...
remote: Enumerating objects: 239, done.
remote: Counting objects: 100% (239/239), done.
remote: Compressing objects: 100% (222/222), done.
remote: Total 239 (delta 77), reused 5 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (239/239), 133.33 MiB | 19.78 MiB/s, done.
Resolving deltas: 100% (77/77), done.


In [3]:
!git -C /content/BMEN-600-Group-19 pull origin main

From https://github.com/chelseataylor1/BMEN-600-Group-19
 * branch            main       -> FETCH_HEAD
Already up to date.


In [4]:
from pathlib import Path
import zipfile

# Locate the output of Notebook 2.
zip_path = Path(
    "/content/BMEN-600-Group-19/data/processed/"
    "OutputFromNotebook2_EHG_Dataset.zip"
)

# Extract the dataset.
with zipfile.ZipFile(zip_path, "r") as zip_file:
    zip_file.extractall("/content")

# Define the locations of the extracted files.
dataset_path = Path("/content/BMEN600_Matched_Trimmed_EHG_Dataset")
recordings_path = dataset_path / "recordings"
metadata_path = dataset_path / "matched_pregnancies.csv"

# Verify the extracted dataset.
print("Dataset extracted successfully!")
print("Recordings:", recordings_path)
print("Metadata:", metadata_path)
print("Number of recordings:", len(list(recordings_path.glob("*.csv"))))

Dataset extracted successfully!
Recordings: /content/BMEN600_Matched_Trimmed_EHG_Dataset/recordings
Metadata: /content/BMEN600_Matched_Trimmed_EHG_Dataset/matched_pregnancies.csv
Number of recordings: 70


In [5]:
# Load the Matched Pregnancy Metadata.
df_matched = pd.read_csv(metadata_path)

# Confirm the metadata loaded successfully.
print("Matched pregnancies loaded:", len(df_matched))

# Preview the metadata.
display(df_matched.head())

Matched pregnancies loaded: 35


,pregnancy_id,age,early_id,early_week,weight_early,later_id,later_week,weight_later,height,placenta,gestation,newborn_weight,outcome,weeks_between
0,17,22,620,23.9,73,1529,31.9,81,168,end,41.1,3930,induced,8.0
1,18,38,656,22.4,77,1385,31.1,82,167,end,41.3,3710,induced,8.7
2,19,26,665,23.1,83,664,30.4,84,165,front,37.1,2860,induced,7.3
3,20,32,757,22.4,68,756,30.6,72,162,end,39.7,3500,induced,8.2
4,21,29,774,25.0,77,775,31.1,83,171,front,41.6,3180,induced,6.1


In [ ]:
# Verify the trimmed EHG recordings
# Find all trimmed EHG recordings.
recording_files = sorted(recordings_path.glob("*.npz"))

# Confirm the number of recordings.
print("Trimmed EHG recordings loaded:", len(recording_files))

# Preview the contents of one recording.
with np.load(recording_files[0]) as recording:
    print("Recording:", recording_files[0].stem)
    print("Available variables:", recording.files)
    print("Sampling frequency:", recording["fs"])
    print("Number of samples:", len(recording["trimmed_S4"]))

Trimmed EHG recordings loaded: 70
Recording: icehg1010
Available variables: ['trimmed_time', 'trimmed_S1', 'trimmed_S2', 'trimmed_S3', 'trimmed_S4', 'fs']
Sampling frequency: 20
Number of samples: 29199


# Create 4 different zip files for each signal type

In [ ]:

# Generate labelled plots for all 70 filtered and trimmed EHG recordings.
# Save each channel separately for visual inspection.

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Define the four EHG channels.
channels = ["S1", "S2", "S3", "S4"]

# Define the output folder.
plots_folder = Path("/content/EHG_Visual_Inspection")
plots_folder.mkdir(parents=True, exist_ok=True)

# Find all trimmed EHG recordings.
recording_files = sorted(recordings_path.glob("*.npz"))

assert len(recording_files) == 70, "Expected 70 EHG recordings."

# Create a separate folder for each channel.
for channel in channels:
    (plots_folder / channel).mkdir(parents=True, exist_ok=True)

# Generate plots for each recording.
for recording_file in recording_files:

    # Extract the unique recording identifier from the filename.
    # Example: icehg799.npz -> icehg799
    record_id = recording_file.stem

    # Load the trimmed recording.
    with np.load(recording_file) as recording:

        new_time = recording["trimmed_time"]

        for channel in channels:

            # Load the filtered and trimmed signal.
            signal = recording[f"trimmed_{channel}"]

            # Create the plot.
            plt.figure(figsize=(12, 4))
            plt.plot(new_time, signal, color="blue", linewidth=0.8)

            # Label the plot with its unique recording identifier.
            plt.title(
                f"Filtered {channel} Signal - Record: {record_id}",
                fontsize=14,
                fontweight="bold"
            )

            plt.xlabel("Time (minutes)")
            plt.ylabel("Amplitude (mV)")
            plt.grid(True)
            plt.tight_layout()

            # Save the plot using the recording identifier and channel.
            # Example: icehg799_S4.png
            plot_path = plots_folder / channel / f"{record_id}_{channel}.png"

            plt.savefig(plot_path, dpi=150)
            plt.close()

print("All 280 labelled EHG plots generated successfully!")

# Verify that each channel contains 70 plots.
for channel in channels:
    saved_plots = list((plots_folder / channel).glob("*.png"))
    print(f"{channel}: {len(saved_plots)} labelled plots")


All 280 labelled EHG plots generated successfully!
S1: 70 labelled plots
S2: 70 labelled plots
S3: 70 labelled plots
S4: 70 labelled plots


In [ ]:

# Create a CSV checklist and ZIP file for each EHG channel.

import pandas as pd
import shutil
import zipfile

zip_files = []

for channel in channels:

    # Locate all 70 plots for this channel.
    channel_folder = plots_folder / channel
    plot_files = sorted(channel_folder.glob("*.png"))

    assert len(plot_files) == 70, (
        f"Expected 70 {channel} plots, found {len(plot_files)}."
    )

    # Create a checklist containing the recording identifiers.
    checklist = pd.DataFrame({
        "Recording ID": [
            plot.stem.rsplit("_", 1)[0] for plot in plot_files
        ],
        "Plot Filename": [
            plot.name for plot in plot_files
        ],
        "Pass/Fail": [""] * len(plot_files),
        "Visual Inspection Notes": [""] * len(plot_files)
    })

    # Save the CSV inside the channel folder.
    csv_path = channel_folder / f"{channel}_Visual_Inspection_Checklist.csv"
    checklist.to_csv(csv_path, index=False)

    # Create a ZIP containing the 70 plots and CSV checklist.
    zip_path = shutil.make_archive(
        f"/content/EHG_Visual_Inspection_{channel}",
        "zip",
        root_dir=plots_folder,
        base_dir=channel
    )

    zip_files.append(zip_path)

    # Verify the ZIP contents.
    with zipfile.ZipFile(zip_path, "r") as archive:
        png_files = [
            f for f in archive.namelist() if f.endswith(".png")
        ]
        csv_files = [
            f for f in archive.namelist() if f.endswith(".csv")
        ]

        assert len(png_files) == 70, "Incorrect number of plots."
        assert len(csv_files) == 1, "Incorrect number of CSV files."

    print(f"{channel}: 70 plots + 1 CSV checklist saved.")

print("\nAll four ZIP files created successfully!")


S1: 70 plots + 1 CSV checklist saved.
S2: 70 plots + 1 CSV checklist saved.
S3: 70 plots + 1 CSV checklist saved.
S4: 70 plots + 1 CSV checklist saved.

All four ZIP files created successfully!


In [ ]:
from google.colab import files

for zip_path in zip_files:
    files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>